In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development.

In [4]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development.
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [5]:
data

,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


In [6]:
from sklearn.model_selection import train_test_split

X = data['text']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

In [8]:
import nltk

In [9]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Data Preprocessing

In [10]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [11]:
# Imports and a check at the raw text
import re

sample = data['text'][0]
print(sample[:1000])

DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL I AM MIKE CHUKWU , THE MANAGER, BILLS AND EXCHANGE AT THE FOREIGN REMITTANCE DEPARTMENT OF THE ZENITH INTERNATIONAL BANK PLC. I AM WRITING THIS LETTER TO ASK FOR YOUR SUPPORT AND COOPERATION TO CARRY OUT THIS BUSINESS OPPORTUNITY IN MY DEPARTMENT. WE DISCOVERED AN ABANDONED SUM OF $15,000,000.00 (FIFTEEN MILLION UNITED STATES DOLLARS ONLY) IN AN ACCOUNT THAT BELONGS TO ONE OF OUR FOREIGN CUSTOMERS WHO DIED ALONG WITH HIS ENTIRE FAMILY OF A WIFE AND TWO CHILDREN IN NOVEMBER 1997 IN A PLANE CRASH. SINCE WE HEARD OF HIS DEATH, WE HAVE BEEN EXPECTING HIS NEXT-OF-KIN TO COME OVER AND PUT CLAIMS FOR HIS MONEY AS THE HEIR,BECAUSE WE CANNOT RELEASE THE FUND FROM HIS ACCOUNT UNLESS SOMEONE APPLIES FOR CLAIM AS THE NEXT-OF-KIN TO THE DECEASED AS INDICATED IN OUR BANKING GUIDELINES. UNFORTUNATELY, NEITHER THEIR FAMILY MEMBER NOR DISTANT RELATIVE HAS EVER APPEARED TO CLAIM THE SAID FUND. UPON THIS DISCOVERY,I AND OTHER OFFICIALS IN MY DEPARTMENT HAVE 

In [12]:
# Remove inline JavaScript/CSS


# Remove <script>...</script> blocks (inline JS)
data['text'] = data['text'].apply(lambda x: re.sub(r'<script.*?>.*?</script>', '', x, flags=re.DOTALL))


# Remove <style>...</style> blocks (inline CSS)
data['text'] = data['text'].apply(lambda x: re.sub(r'<style.*?>.*?</style>', '', x, flags=re.DOTALL))

In [13]:
# Remove html comments

data['text'] = data['text'].apply(lambda x: re.sub(r'<!--.*?-->', '', x, flags=re.DOTALL))

In [14]:
# Remove all remaining HTML tags

data['text'] = data['text'].apply(lambda x: re.sub(r'<.*?>', '', x))

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters

- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [15]:
# Remove the special characters

data['text'] = data['text'].apply(lambda x: re.sub(r'\W', ' ', x))

In [16]:
# Remove numbers

data['text'] = data['text'].apply(lambda x: re.sub(r'\d', ' ', x))

In [17]:
# Remove all single characters

data['text'] = data['text'].apply(lambda x: re.sub(r'\s+[a-zA-Z]\s+', ' ', x))

In [18]:
# Remove a single character at the very start of the string

data['text'] = data['text'].apply(lambda x: re.sub(r'^[a-zA-Z]\s+', ' ', x))

In [19]:
# Substitute multiple spaces with single space

data['text'] = data['text'].apply(lambda x: re.sub(r'\s+', ' ', x, flags=re.I))

In [20]:
# Remove prefixed 'b'

data['text'] = data['text'].apply(lambda x: re.sub(r'^b\s+', '', x))

In [21]:
# Lowercase everything

data['text'] = data['text'].apply(lambda x: x.lower())

## Now let's work on removing stopwords
Remove the stopwords.

In [22]:
data['text'] = data['text'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stopwords.words("english")])
)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [26]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')  # WordNet's multilingual mapping, sometimes required as a dependency

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [27]:
from nltk.tokenize import word_tokenize

data['tokens'] = data['text'].apply(lambda x: word_tokenize(x))
print(data['tokens'][0][:20])

['dear', 'sir', 'strictly', 'private', 'business', 'proposal', 'mike', 'chukwu', 'manager', 'bills', 'exchange', 'foreign', 'remittance', 'department', 'zenith', 'international', 'bank', 'plc', 'writing', 'letter']


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [29]:
ham_data = data[data['label'] == 'ham']
spam_data = data[data['label'] == 'spam']

print(ham_data.shape)
print(spam_data.shape)

(0, 3)
(0, 3)


In [30]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

In [36]:
print(data[data['label'] == 0]['text'].head(3))
print("---")
print(data[data['label'] == 1]['text'].head(3))

1                                                     
2    nora cheryl emailed dozens memos haiti weekend...
4                                                  fyi
Name: text, dtype: object
---
0    dear sir strictly private business proposal mi...
3    dear sir fmadam know proposal might surprise e...
6    dear sir engr ugo nzego engineering stores dep...
Name: text, dtype: object


In [37]:
ham_data = data[data['label'] == 0]
spam_data = data[data['label'] == 1]

print(ham_data.shape)
print(spam_data.shape)

(558, 3)
(442, 3)


In [38]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
ham_counts = vectorizer.fit_transform(ham_data['text'])

ham_word_freq = ham_counts.sum(axis=0)
ham_words_freq = [(word, ham_word_freq[0, idx]) for word, idx in vectorizer.vocabulary_.items()]
ham_words_freq = sorted(ham_words_freq, key=lambda x: x[1], reverse=True)

print(ham_words_freq[:10])

[('pm', np.int64(127)), ('state', np.int64(113)), ('would', np.int64(107)), ('president', np.int64(98)), ('mr', np.int64(86)), ('obama', np.int64(84)), ('percent', np.int64(81)), ('call', np.int64(78)), ('us', np.int64(76)), ('secretary', np.int64(76))]


In [39]:
vectorizer2 = CountVectorizer()
spam_counts = vectorizer2.fit_transform(spam_data['text'])

spam_word_freq = spam_counts.sum(axis=0)
spam_words_freq = [(word, spam_word_freq[0, idx]) for word, idx in vectorizer2.vocabulary_.items()]
spam_words_freq = sorted(spam_words_freq, key=lambda x: x[1], reverse=True)

print(spam_words_freq[:10])

[('money', np.int64(981)), ('account', np.int64(836)), ('bank', np.int64(780)), ('us', np.int64(734)), ('transaction', np.int64(538)), ('business', np.int64(514)), ('country', np.int64(475)), ('nbsp', np.int64(475)), ('fund', np.int64(474)), ('million', np.int64(439))]


## Extra features

In [45]:
data['preprocessed_text'] = data['text']

In [46]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(
    data,
    test_size=0.2,
    random_state=0,
    stratify=data['label']
)

print(data_train.shape)
print(data_val.shape)

(800, 4)
(200, 4)


In [47]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x))

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x))

data_train.head()

,text,label,tokens,preprocessed_text,money_mark,suspicious_words,text_len
886,john ryan susan mcnamara corrine ebeling delan...,0,"[john, ryan, susan, mcnamara, corrine, ebeling...",john ryan susan mcnamara corrine ebeling delan...,0,1,3425
419,miss jannet martins abidjan ivory coast greeti...,1,"[miss, jannet, martins, abidjan, ivory, coast,...",miss jannet martins abidjan ivory coast greeti...,0,1,1085
939,fyi,0,[fyi],fyi,0,0,3
917,talk,0,[talk],talk,0,0,4
552,ok,0,[ok],ok,0,0,2


## How would you create a Bag of Words with the CountVectorizer method?

In [48]:
X_train_bow = vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_bow = vectorizer.transform(data_val['preprocessed_text'])

print(X_train_bow.shape)
print(X_val_bow.shape)

(800, 23571)
(200, 23571)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [50]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

In [51]:
X_tfidf = tfidf_vectorizer.fit_transform(data['preprocessed_text'])

In [52]:
print(X_tfidf.shape)

(1000, 31056)


### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code